In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [22]:
df=pd.read_csv('../data/raw/fake_job_postings.csv')

In [23]:
df['has_company_logo'] = df['has_company_logo'].fillna(0)
df['has_questions'] = df['has_questions'].fillna(0)
df['telecommuting'] = df['telecommuting'].fillna(0)
df['employment_type']=df['employment_type'].fillna('Unknown')
df['employment_type'] = df['employment_type'].str.lower()

In [24]:
df['employment_type'].unique()

array(['other', 'full-time', 'unknown', 'part-time', 'contract',
       'temporary'], dtype=object)

In [25]:
employment_dummies = pd.get_dummies(
    df['employment_type'],
)

df = pd.concat([df, employment_dummies], axis=1)


In [26]:
df.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,...,required_education,industry,function,fraudulent,contract,full-time,other,part-time,temporary,unknown
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,...,NaN,NaN,Marketing,0,False,False,True,False,False,False
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,...,NaN,Marketing and Advertising,Customer Service,0,False,True,False,False,False,False
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,...,NaN,NaN,NaN,0,False,False,False,False,False,True
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,...,Bachelor's Degree,Computer Software,Sales,0,False,True,False,False,False,False
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,...,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0,False,True,False,False,False,False


In [28]:
df['company_profile_length'] = df['company_profile'].fillna('').apply(len)

In [31]:
import spacy
nlp = spacy.load("en_core_web_sm")

df['org_count'] = 0
df['gpe_count'] = 0

for i in range(len(df)):
    text = df.loc[i, 'company_profile']
    
    if pd.isna(text):
        continue
    
    doc = nlp(text)
    
    for ent in doc.ents:
        if ent.label_ == 'ORG':
            df.loc[i, 'org_count'] += 1
        elif ent.label_ == 'GPE':
            df.loc[i, 'gpe_count'] += 1

In [32]:
df['company_profile_length_log'] = np.log1p(df['company_profile_length'])
df['org_count_log'] = np.log1p(df['org_count'])
df['gpe_count_log'] = np.log1p(df['gpe_count'])